# Pruebas de la implementación del notebook

Ejecutar todo. Se cargan solo las celdas etiquetadas `implementation` del notebook 01: definiciones, sin lanzar su flujo ni leer radiografías reales. No se importan módulos `.py` locales.

Las pruebas usan archivos sintéticos temporales: lectores genérico/CheXpert/NIH, exclusiones, duplicados, split por paciente, test oficial, reproducibilidad, uint16 y estadísticas de train independientes de test. No evalúan un modelo médico.


In [ ]:
import json
from pathlib import Path
import unittest

candidates = [Path.cwd(), *Path.cwd().parents]
NOTEBOOK = next((p / 'notebooks/01_preparacion_radiografias.ipynb' for p in candidates
                 if (p / 'notebooks/01_preparacion_radiografias.ipynb').is_file()),
                Path.cwd() / '01_preparacion_radiografias.ipynb')
if not NOTEBOOK.is_file():
    raise FileNotFoundError('Colocar ambos notebooks juntos o abrir desde la raíz del repositorio.')
source_notebook = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
loaded = 0
for cell in source_notebook['cells']:
    if cell['cell_type'] == 'code' and 'implementation' in cell.get('metadata', {}).get('tags', []):
        source = cell['source']
        source = source if isinstance(source, str) else ''.join(source)
        exec(compile(source, str(NOTEBOOK), 'exec'), globals())
        loaded += 1
print('Celdas de implementación cargadas:', loaded)


In [ ]:
"""Pruebas del flujo completo y de las barreras contra fuga de datos."""

import csv

import json

import tempfile

import unittest

from pathlib import Path

import numpy as np

from PIL import Image

class PreparationTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)
        self.raw = generate(self.root / 'raw')
        self.config = {'dataset': 'generic', 'target': 'target',
                       'metadata': str(self.raw / 'metadata.csv'), 'image_root': str(self.raw),
                       'output': str(self.root / 'processed'), 'image_size': 32,
                       'normalization': 'unit', 'seed': 42, 'ratios': [0.7, 0.15, 0.15]}

    def run_pipeline(self):
        path = self.root / 'config.json'
        path.write_text(json.dumps(self.config), encoding='utf-8')
        return prepare(path)

    def rows(self):
        with (self.raw / 'metadata.csv').open(newline='', encoding='utf-8') as stream:
            return list(csv.DictReader(stream))

    def write_rows(self, rows):
        with (self.raw / 'metadata.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

    def test_end_to_end_and_repeatability(self):
        out = self.run_pipeline()
        patients, total = [], 0
        for name in ('train', 'val', 'test'):
            with (out / f'{name}.csv').open(newline='', encoding='utf-8') as stream:
                rows = list(csv.DictReader(stream))
            x = np.load(out / f'X_{name}.npy', mmap_mode='r', allow_pickle=False)
            y = np.load(out / f'y_{name}.npy', allow_pickle=False)
            self.assertEqual(x.shape, (len(rows), 32, 32))
            self.assertEqual(x.dtype, np.float32)
            self.assertTrue(np.isfinite(x).all())
            self.assertGreaterEqual(float(x.min()), 0)
            self.assertLessEqual(float(x.max()), 1)
            np.testing.assert_array_equal(y, [int(r['target']) for r in rows])
            self.assertEqual(set(y), {0, 1})
            patients.append({r['patient_id'] for r in rows})
            total += len(rows)
            del x
        self.assertEqual(total, 120)
        self.assertFalse(patients[0] & patients[1] | patients[0] & patients[2] | patients[1] & patients[2])
        self.config['output'] = str(self.root / 'repeat')
        repeated = self.run_pipeline()
        for name in ('train', 'val', 'test'):
            self.assertEqual((out/f'{name}.csv').read_bytes(), (repeated/f'{name}.csv').read_bytes())
            self.assertEqual((out/f'X_{name}.npy').read_bytes(), (repeated/f'X_{name}.npy').read_bytes())
        with self.assertRaisesRegex(ValueError, 'ya existe'):
            self.run_pipeline()

    def test_bad_labels_files_and_duplicates_are_accounted(self):
        rows = self.rows()
        additions = [dict(rows[0], target='-1'), dict(rows[1], target=''),
                     dict(rows[2], image_path='images/missing.png'),
                     dict(rows[3], image_path='images/broken.png'), dict(rows[4])]
        (self.raw/'images/broken.png').write_bytes(b'not an image')
        self.write_rows(rows + additions)
        out = self.run_pipeline()
        report = json.loads((out/'report.json').read_text())
        for reason in ('uncertain_label', 'missing_label', 'missing_file', 'unreadable_image', 'duplicate_image'):
            self.assertEqual(report['exclusion_reasons'][reason], 1)
        self.assertEqual(report['source_rows'], report['accepted_images'] + report['excluded_rows'])

    def test_duplicate_across_patients_or_conflicting_labels_removes_both(self):
        rows = self.rows()
        self.write_rows(rows + [dict(rows[0], patient_id='other'), dict(rows[1], target='1')])
        records, excluded, _ = read_metadata(self.config)
        kept = audit(records, excluded)
        self.assertEqual(len(kept), 118)
        self.assertEqual(sum(r['reason'] == 'duplicate_across_patients' for r in excluded), 2)
        self.assertEqual(sum(r['reason'] == 'duplicate_conflicting_label' for r in excluded), 2)

    def test_explicit_patient_leakage_is_rejected(self):
        rows = self.rows()
        for i, row in enumerate(rows):
            row['split'] = ('train', 'val', 'test')[i % 3]
        self.write_rows(rows)
        with self.assertRaisesRegex(ValueError, 'Fuga'):
            self.run_pipeline()
        self.assertFalse((self.root/'processed').exists())
        self.assertTrue((self.root/'processed.audit.csv').exists())

    def test_train_statistics_do_not_depend_on_test(self):
        rows = self.rows()
        for row in rows:
            patient = int(row['patient_id'][1:])
            row['split'] = 'train' if patient < 40 else 'val' if patient < 50 else 'test'
        self.write_rows(rows)
        self.config['normalization'] = 'zscore'
        first = self.run_pipeline()
        stats = json.loads((first/'preprocessing.json').read_text())
        rng = np.random.default_rng(888)
        for row in rows:
            if row['split'] == 'test':
                Image.fromarray(rng.integers(200, 256, (48, 40), dtype=np.uint8)).save(self.raw/row['image_path'])
        self.config['output'] = str(self.root/'second')
        second = self.run_pipeline()
        self.assertEqual(stats, json.loads((second/'preprocessing.json').read_text()))
        a = np.load(first/'X_train.npy')
        b = np.load(second/'X_train.npy')
        np.testing.assert_array_equal(a, b)
        self.assertAlmostEqual(float(a.mean()), 0, places=5)
        self.assertAlmostEqual(float(a.std()), 1, places=5)

    def test_uint16_preserves_intensity(self):
        path = self.raw/'images/16bit.png'
        values = np.array([[0, 1000], [32768, 65535]], dtype=np.uint16)
        Image.fromarray(values).save(path)
        data, info = read_image(path)
        self.assertEqual(info['intensity_scale'], 65535)
        np.testing.assert_allclose(data, values.astype(np.float32)/65535)

    def test_resize_preserves_aspect_with_padding(self):
        path = self.raw/'images/wide.png'
        Image.fromarray(np.full((10, 20), 255, dtype=np.uint8)).save(path)
        result = transform(path, 32)
        self.assertTrue(np.all(result[:8] == 0))
        self.assertTrue(np.all(result[8:24] == 1))
        self.assertTrue(np.all(result[24:] == 0))

    def test_partial_explicit_splits_and_path_escape_rejected(self):
        rows = self.rows()
        for i, row in enumerate(rows):
            row['split'] = 'train' if i == 0 else ''
        self.write_rows(rows)
        with self.assertRaisesRegex(ValueError, 'split debe estar completa'):
            read_metadata(self.config)
        rows = [dict(image_path='../outside.png', patient_id='p', target='1')]
        self.write_rows(rows)
        records, excluded, _ = read_metadata(self.config)
        self.assertEqual(records, [])
        self.assertEqual(excluded[0]['reason'], 'path_outside_image_root')

    def test_chexpert_adapter_excludes_uncertainty(self):
        base = self.raw/'CheXpert-v1.0-small/train/patient00001/study1'
        base.mkdir(parents=True)
        Image.fromarray(np.arange(100, dtype=np.uint8).reshape(10, 10)).save(base/'view1.png')
        self.write_rows([{'Path': 'CheXpert-v1.0-small/train/patient00001/study1/view1.png',
                          'Frontal/Lateral': 'Frontal', 'Effusion': '1'},
                         {'Path': 'missing.png', 'Frontal/Lateral': 'Frontal', 'Effusion': '-1'}])
        self.config.update(dataset='chexpert', target='Effusion', frontal_only=True)
        records, excluded, _ = read_metadata(self.config)
        kept = audit(records, excluded)
        self.assertEqual(len(kept), 1)
        self.assertEqual(kept[0]['patient_id'], 'patient00001')
        self.assertEqual(excluded[0]['reason'], 'uncertain_label')

    def test_nih_adapter_and_ambiguous_filename(self):
        original = self.rows()[0]['image_path']
        self.write_rows([{'Image Index': Path(original).name, 'Patient ID': '123',
                          'Finding Labels': 'Effusion|Atelectasis', 'View Position': 'PA'}])
        self.config.update(dataset='nih', target='Effusion', frontal_only=True)
        records, excluded, _ = read_metadata(self.config)
        self.assertEqual(len(audit(records, excluded)), 1)
        other = self.raw/'other'
        other.mkdir()
        (other/Path(original).name).write_bytes((self.raw/original).read_bytes())
        records, excluded, _ = read_metadata(self.config)
        self.assertFalse(records)
        self.assertEqual(excluded[0]['reason'], 'ambiguous_filename')

    def test_nih_official_test_is_preserved(self):
        rows = self.rows()
        self.write_rows([{'Image Index': Path(r['image_path']).name, 'Patient ID': r['patient_id'],
                          'Finding Labels': 'Effusion' if r['target'] else 'No Finding',
                          'View Position': 'PA'} for r in
                         [dict(r, target=int(r['target'])) for r in rows]])
        train = [Path(r['image_path']).name for r in rows if int(r['patient_id'][1:]) < 50]
        test = [Path(r['image_path']).name for r in rows if int(r['patient_id'][1:]) >= 50]
        (self.raw/'train_val_list.txt').write_text('\n'.join(train))
        (self.raw/'test_list.txt').write_text('\n'.join(test))
        self.config.update(dataset='nih', target='Effusion', frontal_only=True)
        out = self.run_pipeline()
        with (out/'test.csv').open(newline='', encoding='utf-8') as stream:
            exported = list(csv.DictReader(stream))
        self.assertEqual({Path(r['image_path']).name for r in exported}, set(test))
        report = json.loads((out/'report.json').read_text())
        self.assertEqual(report['split_method'], 'explicit_test_patient_validation_from_train')

    def test_insufficient_class_support_stops_export(self):
        rows = self.rows()
        for row in rows:
            row['target'] = '0'
        self.write_rows(rows)
        with self.assertRaisesRegex(ValueError, 'cada clase'):
            self.run_pipeline()
        self.assertFalse((self.root/'processed').exists())


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(PreparationTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Hay pruebas fallidas: revisar los mensajes anteriores.'
print(f'Correcto: {result.testsRun} pruebas completadas.')
